# KRAS Allele-Specific CRISPR Dependency — Box Plots

This notebook generates publication-quality box plots comparing CRISPR gene effect scores (Chronos) across four KRAS genotype groups in colorectal adenocarcinoma (COAD): **G12D**, **G12V**, **A146T**, and **WT**.

It is intended to visualize the top hits from `anova.ipynb`. The data loading and group-definition logic is self-contained here so this notebook can be run independently.

**Data:** DepMap Public 24Q2  
**Data DOI:** [10.25452/figshare.plus.27993248.v1](https://doi.org/10.25452/figshare.plus.27993248.v1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from pathlib import Path

In [ ]:
# --- Configure paths ---
DATA_DIR = Path("/path/to/depmap_24Q2")

MODEL_PATH  = DATA_DIR / "Model.csv"
OSM_PATH    = DATA_DIR / "OmicsSomaticMutations.csv"
EFFECT_PATH = DATA_DIR / "CRISPRGeneEffect.csv"

# Genes to plot — top hits from the ANOVA (most essential in G12D)
# Edit this list to explore other genes of interest
GENES_TO_PLOT = ['RPS7', 'POLR2H', 'MASTL', 'RPL13', 'KRAS']

CANCER_TYPE  = "COAD"
GENE         = "KRAS"
KRAS_ALLELES = ['p.G12D', 'p.G12V', 'p.A146T']

## 1. Load Data and Define Groups

In [ ]:
model_df  = pd.read_csv(MODEL_PATH,  header=0, index_col=0)
osm_df    = pd.read_csv(OSM_PATH,    header=0, low_memory=False)
effect_df = pd.read_csv(EFFECT_PATH, header=0, index_col=0)

# Strip Entrez IDs from column names ("KRAS (3845)" -> "KRAS")
effect_df.columns = [col.split(' ')[0] for col in effect_df.columns]

print(f"Loaded {effect_df.shape[0]} lines x {effect_df.shape[1]} genes")

In [ ]:
# All COAD models
coad_models = model_df[model_df['OncotreeCode'] == CANCER_TYPE].reset_index()[['ModelID']]
osm_coad    = coad_models.merge(osm_df, on='ModelID', how='inner')
kras_osm    = osm_coad[osm_coad['HugoSymbol'] == GENE]

# Per-allele groups
mutation_groups = {}
for allele in KRAS_ALLELES:
    ids = kras_osm[kras_osm['ProteinChange'] == allele][['ModelID']].drop_duplicates()
    mutation_groups[allele] = ids.merge(effect_df, on='ModelID', how='inner').set_index('ModelID')

# WT group: no KRAS mutation at all
all_kras_ids = kras_osm['ModelID'].unique()
wt_ids = coad_models[~coad_models['ModelID'].isin(all_kras_ids)]
mutation_groups['WT'] = wt_ids.merge(effect_df, on='ModelID', how='inner').set_index('ModelID')

group_order  = KRAS_ALLELES + ['WT']
group_labels = ['G12D', 'G12V', 'A146T', 'WT']

for key, label in zip(group_order, group_labels):
    print(f"{label}: {len(mutation_groups[key])} lines")

## 2. Box Plots with Significance Annotations

Each panel shows the distribution of CRISPR effect scores (Chronos) for one gene across the four genotype groups. Individual data points are overlaid as a strip plot to show the actual sample sizes.

The dashed red line marks the conventional DepMap essentiality threshold (~-0.5). Pairwise Welch's t-tests are run between each mutant allele and WT; significance bars are shown for comparisons passing an FDR-corrected threshold of 0.05.

In [ ]:
def add_significance_bars(ax, data_by_group, group_labels, wt_idx=3):
    """Add significance bars comparing each mutant group to WT."""
    n_groups = len(data_by_group)
    mutant_indices = [i for i in range(n_groups) if i != wt_idx]

    p_vals = []
    pairs  = []
    for i in mutant_indices:
        a = data_by_group[i]
        b = data_by_group[wt_idx]
        if len(a) > 1 and len(b) > 1:
            _, p = scipy.stats.ttest_ind(a, b, equal_var=False)
            p_vals.append(p)
            pairs.append((i, wt_idx))

    if not p_vals:
        return

    # BH correction across the comparisons for this gene
    _, fdr_vals, _, _ = multipletests(p_vals, method='fdr_bh')

    # Get current y limits to position bars above data
    y_max = max(np.nanmax(d) for d in data_by_group if len(d) > 0)
    y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
    bar_height = y_range * 0.07
    bar_tips   = y_range * 0.02

    for idx, ((i, j), fdr) in enumerate(zip(pairs, fdr_vals)):
        if fdr >= 0.05:
            continue
        star = '***' if fdr < 0.001 else ('**' if fdr < 0.01 else '*')

        y = y_max + bar_height * (idx + 1)
        x1, x2 = i + 1, j + 1

        ax.plot([x1, x1, x2, x2],
                [y - bar_tips, y, y, y - bar_tips],
                lw=1.0, color='#444444')
        ax.text((x1 + x2) / 2, y + bar_tips * 0.5, star,
                ha='center', va='bottom', fontsize=10, color='#444444')

In [ ]:
colors = ['#E07B54', '#5B8DB8', '#6AAB6E', '#AAAAAA']

n_genes = len(GENES_TO_PLOT)
fig, axes = plt.subplots(1, n_genes, figsize=(3.2 * n_genes, 5), sharey=False)
if n_genes == 1:
    axes = [axes]

np.random.seed(42)

for ax, gene in zip(axes, GENES_TO_PLOT):
    if gene not in effect_df.columns:
        ax.set_title(f'{gene}\n(not found)', fontsize=10)
        continue

    data_by_group = [
        mutation_groups[k][gene].dropna().values
        for k in group_order
    ]

    bp = ax.boxplot(
        data_by_group,
        patch_artist=True,
        widths=0.45,
        medianprops=dict(color='black', linewidth=2.0),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker='o', markersize=3, alpha=0.4, markeredgewidth=0)
    )

    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)

    # Jittered data points
    for i, (vals, color) in enumerate(zip(data_by_group, colors)):
        if len(vals) == 0:
            continue
        jitter = np.random.uniform(-0.14, 0.14, size=len(vals))
        ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                   color=color, s=26, alpha=0.85, zorder=4,
                   edgecolors='white', linewidths=0.5)

    # Essentiality threshold
    ax.axhline(y=-0.5, color='#CC3333', linestyle='--', linewidth=0.9,
               alpha=0.7, label='Essential threshold (−0.5)')

    ax.set_xticks(range(1, len(group_labels) + 1))
    ax.set_xticklabels(group_labels, fontsize=9)
    ax.set_title(gene, fontsize=12, fontweight='bold', pad=6)

    if ax is axes[0]:
        ax.set_ylabel('CRISPR Gene Effect (Chronos)', fontsize=9)
        ax.legend(fontsize=7.5, loc='upper right', framealpha=0.7)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='y', labelsize=8)

    add_significance_bars(ax, data_by_group, group_labels, wt_idx=3)

fig.suptitle(
    f'CRISPR Gene Effect by KRAS Genotype — {CANCER_TYPE} (DepMap 24Q2)\n'
    f'* FDR<0.05  ** FDR<0.01  *** FDR<0.001  (each mutant vs WT)',
    fontsize=11, y=1.03
)

# Shared legend for group colors
legend_patches = [
    mpatches.Patch(facecolor=c, alpha=0.7, label=l)
    for c, l in zip(colors, group_labels)
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.04), framealpha=0.8)

plt.tight_layout()
plt.savefig("outputs/boxplots_kras_alleles.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/boxplots_kras_alleles.png")

## 3. Interpretation

The box plots confirm that the top ANOVA hits are primarily driven by stronger negative CRISPR scores in **G12D** lines relative to WT. Key observations:

- **RPS7** and **RPL13** (ribosomal subunits) are among the most essential genes in G12D COAD lines, consistent with pathway-level GSEA enrichment of `KEGG_RIBOSOME`
- **POLR2H** (RNA polymerase II subunit) supports the `KEGG_RNA_POLYMERASE` enrichment
- **MASTL** (kinase required for mitotic entry) shows significant variation across alleles, with G12D showing the strongest dependency
- **KRAS** itself is included as a positive control — G12D lines show the most negative KRAS effect score, validating that the screen detects oncogene addiction in the expected direction

The G12V and A146T groups trend in the same direction for most hits but with weaker and less consistent effects, suggesting these dependencies may be partially allele-specific or simply reflect the smaller group sizes.